# 05 — Q3: What can be said definitively? The claims ledgers

Everything `01`-`04b` produced, collapsed into one tier per claim. A claim is a
(program or gene) × (cell group) × (effect) triple. Effects: **PSCA** (PSCA vs TAG72, replicated
across RT strata), **direct RT** (Tu1), **indirect RT** (Tu2), **CAR × RT**, **within-mouse DiD**.

| tier | rule |
|---|---|
| **Definitive** | a second method agrees (GSEA replicated for sets >= 10 genes; gene-level replication in `02` for smaller modules and single genes) **and** the sample-level score test agrees in sign with p < 0.05 **and** it survives leave-one-mouse-out **and** it is not ambient-suspect **and** (for programs) it beats >= 95% of random programs |
| **Supported** | sign-consistent in both strata / backgrounds, and either method significant, but one of the Definitive conditions missing |
| **Suggestive** | one stratum or background only, or an interaction / DiD at q or p < 0.05 |
| **No evidence** | none of the above — printed with the minimum detectable effect, so "absent" and "could not have seen it" stay apart |

The **chemokine ledger** is separate, with ligands and receptors in separate rows. It keeps
two columns apart that must never be merged:

* **observed transcriptional change** — what the data measured;
* **inferred chemotactic potential** — what it would mean if the transcript became protein, a
  gradient and a migrating cell. This column is capped at *Suggestive* without a population
  association, and it never says "recruitment": migration is not measured here.

## 1. Config and inputs

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, itertools
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats

pd.set_option("display.width", 300); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 500); pd.set_option("display.max_colwidth", 60)
_HPC = Path("/coh_labs/yunroseli/Jona/CAR-T"); _LOC = Path("/Users/janzules/Roselab/Spatial/CAR_T")
PROJ = _LOC if _LOC.exists() else _HPC
OUT = PROJ/"results/mhc_cytotox"
MIN_CELLS = 10

CAT = json.load(open(OUT/"programs.json")); PROGRAMS, AXES = CAT["programs"], CAT["axes"]
PB = pd.read_parquet(OUT/"pseudobulk_counts.parquet")
PBM = pd.read_csv(OUT/"pseudobulk_meta.csv").set_index("key").loc[PB.index]
L = np.log2(PB.div(PB.sum(1), axis=0) * 1e6 + 1)
PT = pd.read_csv(OUT/"program_score_contrasts.csv")
SO = pd.read_csv(OUT/"stands_out.csv")
REPL = pd.read_csv(OUT/"de_panel_replication.csv")
PANEL = pd.read_csv(OUT/"de_panel_genes.csv")
AMB = pd.read_csv(OUT/"q1_ambient_estimate.csv")
SENS = pd.read_csv(OUT/"sensitivity_ccl5_xcl1.csv")
EX = pd.read_csv(OUT/"exhaustion_sample_contrasts.csv"); EMDE = pd.read_csv(OUT/"exhaustion_mde.csv")
AXT = pd.read_csv(OUT/"chemokine_axis_table_sc.csv")
RECX = pd.read_csv(OUT/"chem_receptor_fraction_contrasts.csv")
design = pd.read_csv(OUT/"design_samples.csv").set_index("sample")
print("inputs loaded")

GROUPS = ["Cancer_cell", "IFN_stimulated_Mac", "M2_like_Mac", "Intermediate_Mac", "Classical_Mono", "cDC",
          "N1_like_Neu", "NK", "CD8_T", "cytotoxic_lymph", "SPP1_TAM"]
HEAD = ["MHC_I_machinery", "MHC_I_Reactome", "MHC_II_core", "MHC_II_Reactome", "IFNG_Hallmark", "ISG_core",
        "cyto_effector", "cyto_effector_noCcl5", "R01_granzyme_GO", "T_cytotox_GO",
        "exh_inhibitory_receptors", "exh_TFs", "exh_progenitor", "exh_progenitor_noXcl1", "exh_terminal",
        "checkpoint_ligands"]
NON_MYELOID = {"Cancer_cell", "NK", "CD8_T", "cytotoxic_lymph", "cDC", "N1_like_Neu"}

inputs loaded


## 2. Is the tumour MHC-II interaction contamination? The ambient fraction, tested the same way

`02` found a CAR × RT interaction on tumour-cell MHC-II (`H2-Ab1`, `H2-Eb1`, `Cd74` ~ +2 log2),
and `01` found tumour MHC-II is mostly ambient (intrinsic fraction 0.04-0.30). If the
**contamination fraction ρ itself** carries the same interaction, the MHC-II result is
contamination. Two tests on the tumour pseudobulk: the interaction on log ρ, and on MHC-II
after subtracting the ambient expectation (ρ × macrophage CPM).

In [2]:
def two_arm_null(v, A, B, site):
    meta = design.loc[[s for s in v.index if s in design.index]]
    idx = [i for i in meta.index[meta.arm.isin([A, B]) & (meta.site == site)] if np.isfinite(v.get(i, np.nan))]
    obs = (meta.loc[idx, "arm"] == A).values; k, n = int(obs.sum()), len(idx)
    if k < 2 or n - k < 2: return None
    x = v.loc[idx].values
    out = [x[obs].mean() - x[~obs].mean()]
    for c in itertools.combinations(range(n), k):
        m = np.isin(np.arange(n), c)
        if not (m == obs).all(): out.append(x[m].mean() - x[~m].mean())
    return np.array(out)


def p2(a): return float((np.abs(a) >= abs(a[0]) - 1e-12).mean())


def carxrt(v):
    """CAR x RT at each site and pooled: (RT effect in PSCA) - (RT effect in TAG72)."""
    out = {}
    for s in (1, 2):
        a = two_arm_null(v, "RTCyPSCA", "CyPSCA", s); b = two_arm_null(v, "RTCyT72", "CyT72", s)
        if a is not None and b is not None:
            it = (a[:, None] - b[None, :]).ravel(); out[f"Tu{s}"] = (it[0], p2(it))
    return out


rho = AMB[AMB.compartment == "Cancer_cell"].groupby("sample").rho.first()
mac_cpm = PB.loc[PBM.group.eq("mac_all").values].pipe(lambda d: d.div(d.sum(1), axis=0) * 1e6)
mac_cpm.index = PBM.loc[mac_cpm.index, "sample"].values
tum = PBM[(PBM.group == "Cancer_cell") & (PBM.n_cells >= MIN_CELLS)]
tum_cpm = PB.loc[tum.index].pipe(lambda d: d.div(d.sum(1), axis=0) * 1e6); tum_cpm.index = tum["sample"].values
rows = [dict(measure="log2 rho (macrophage contamination)", **{k: f"{v[0]:+.2f} (p={v[1]:.3f})"
                                                              for k, v in carxrt(np.log2(rho)).items()})]
PC = 10     # CPM pseudocount: subtraction can leave near-zero CPM, and log2(x + 1) would turn that into
            # a +10 log2 "effect" from noise
CL = pd.read_csv(OUT/"q1_tumour_clean_vs_rest.csv").set_index("sample")
for g in ["H2-Ab1", "H2-Eb1", "Cd74", "H2-Aa", "H2-K1", "B2m"]:
    obs_ = np.log2(tum_cpm[g] + PC)
    corr = np.log2((tum_cpm[g] - rho.reindex(tum_cpm.index) * mac_cpm.loc[tum_cpm.index, g]).clip(lower=0) + PC)
    rows.append(dict(measure=f"{g} observed", **{k: f"{v[0]:+.2f} (p={v[1]:.3f})" for k, v in carxrt(obs_).items()}))
    rows.append(dict(measure=f"{g} ambient-subtracted", **{k: f"{v[0]:+.2f} (p={v[1]:.3f})" for k, v in carxrt(corr).items()}))
    if f"{g}_clean" in CL:   # tumour cells with ZERO macrophage-marker reads (from 01): the cleanest test
        rows.append(dict(measure=f"{g} clean tumour cells only", **{k: f"{v[0]:+.3f} (p={v[1]:.3f})"
                                                                   for k, v in carxrt(CL[f"{g}_clean"]).items()}))
AMBX = pd.DataFrame(rows)
AMBX.to_csv(OUT/"ledger_tumour_mhc_ambient_check.csv", index=False)
print(AMBX.to_string(index=False))
r_ = stats.spearmanr(np.log2(rho.reindex(tum_cpm.index)), np.log2(tum_cpm["H2-Ab1"] + 1)).statistic
print(f"\nacross samples, tumour H2-Ab1 vs contamination rho: Spearman {r_:+.2f}")
print("READ: if log2 rho shows the interaction and the ambient-subtracted / clean-cell MHC-II does not, the\n"
      "tumour MHC-II CAR x RT result is macrophage RNA. If it survives in clean cells, part of it is the tumour's\n"
      "own (IFN-gamma-induced MHC-II on tumour cells is biologically possible; Ciita is ~50% intrinsic here).\n"
      "Clean-cell values are mean log-norm per cell, so their effect sizes are on a different scale.")

                            measure              Tu1              Tu2
log2 rho (macrophage contamination)  +1.45 (p=0.165)  +0.31 (p=0.755)
                    H2-Ab1 observed  +2.12 (p=0.045)  +2.38 (p=0.025)
          H2-Ab1 ambient-subtracted  +2.85 (p=0.250)  +3.08 (p=0.020)
     H2-Ab1 clean tumour cells only +0.151 (p=0.415) +0.095 (p=0.445)
                    H2-Eb1 observed  +2.36 (p=0.055)  +2.83 (p=0.030)
          H2-Eb1 ambient-subtracted  +3.33 (p=0.180)  +4.74 (p=0.005)
     H2-Eb1 clean tumour cells only +0.357 (p=0.100) +0.135 (p=0.370)
                      Cd74 observed  +2.10 (p=0.040)  +2.80 (p=0.045)
            Cd74 ambient-subtracted  +2.39 (p=0.395)  +6.05 (p=0.005)
       Cd74 clean tumour cells only +0.163 (p=0.655) +0.216 (p=0.260)
                     H2-Aa observed  +2.03 (p=0.080)  +2.02 (p=0.105)
           H2-Aa ambient-subtracted  +2.93 (p=0.235)  +3.80 (p=0.025)
      H2-Aa clean tumour cells only +0.259 (p=0.365) +0.078 (p=0.680)
                    

## 3. Leave-one-mouse-out for program scores

The `02` LOO was gene-level. Here, for every program × group × effect, each usable sample is
dropped in turn and the stratum deltas recomputed: a claim survives if no single animal flips
the sign of either stratum.

In [3]:
def group_scores(group):
    d = PBM[(PBM.group == group) & (PBM.n_cells >= MIN_CELLS)]
    Lg = L.loc[d.index]; expressed = Lg.columns[(Lg.mean() >= 1.0).values]
    tr = (d.arm != "NoTx").values
    Z = (Lg[expressed] - Lg.loc[tr, expressed].mean()) / Lg.loc[tr, expressed].std().replace(0, np.nan)
    Z = Z.dropna(axis=1); Z.index = d["sample"].values
    return pd.DataFrame({k: Z[[g for g in v["genes"] if g in Z]].mean(1) for k, v in PROGRAMS.items()
                         if sum(g in Z for g in v["genes"]) >= (1 if v["family"] == "chemokine" else 3)})


STRATA = {"PSCA": [("CyPSCA", "CyT72", (1, 2)), ("RTCyPSCA", "RTCyT72", (1, 2))],
          "direct RT": [("RTCyPSCA", "CyPSCA", (1,)), ("RTCyT72", "CyT72", (1,))],
          "indirect RT": [("RTCyPSCA", "CyPSCA", (2,)), ("RTCyT72", "CyT72", (2,))]}


def stratum_delta(v, A, B, sites):
    ds = []
    for s in sites:
        idx = [i for i in v.index if design.loc[i, "site"] == s]
        a = v[[i for i in idx if design.loc[i, "arm"] == A]]; b = v[[i for i in idx if design.loc[i, "arm"] == B]]
        if len(a) and len(b): ds.append(a.mean() - b.mean())
    return np.mean(ds) if ds else np.nan


loo = []
for g in GROUPS:
    S = group_scores(g)
    for prog in [p for p in HEAD + [f"{a}_{r}" for a in AXES for r in ("ligands", "receptors")] if p in S]:
        v = S[prog].dropna()
        for eff, strata in STRATA.items():
            base = [stratum_delta(v, *st) for st in strata]
            flips, n = 0, 0
            for s in v.index:
                if design.loc[s, "arm"] == "NoTx": continue
                vv = v.drop(index=s); n += 1
                new = [stratum_delta(vv, *st) for st in strata]
                flips += any(np.sign(a) != np.sign(b) for a, b in zip(base, new) if np.isfinite(a) and np.isfinite(b))
            loo.append(dict(group=g, program=prog, effect=eff, loo_flips=flips, loo_n=n))
LOO = pd.DataFrame(loo)
print(f"program-score LOO computed for {len(LOO)} group x program x effect cells; "
      f"{int((LOO.loo_flips == 0).sum())} survive every deletion")

program-score LOO computed for 813 group x program x effect cells; 277 survive every deletion


## 4. The program ledger (MHC, IFN, cytotoxicity, exhaustion, checkpoint)

In [4]:
amb_genes = AMB.groupby(["compartment", "gene"]).intrinsic_frac.median()


def ambient_flag(group, prog):
    if group not in NON_MYELOID: return False
    gl = [g for g in PROGRAMS[prog]["genes"] if (group, g) in amb_genes.index]
    if not gl: return False
    return float(np.nanmedian([amb_genes[(group, g)] for g in gl])) < 0.5


# minimum detectable main effect in within-arm SD units, 80% power, two-sided 0.05: one 3 v 3 stratum
# detects (t_.975,4 + t_.80,4) * sqrt(2/3) ~ 3.0 SD; averaging two independent strata divides by sqrt(2)
# (~2.1 SD); the PSCA strata already pool two sites (~6 v 5), so its main effect reaches ~1.5 SD.
_k = (stats.t.ppf(0.975, 4) + stats.t.ppf(0.80, 4)) * np.sqrt(2 / 3)
MDE_SD = {"PSCA": round(_k / 2, 1), "direct RT": round(_k / np.sqrt(2), 1),
          "indirect RT": round(_k / np.sqrt(2), 1), "DiD": round(_k / np.sqrt(2), 1)}
print("MDE (SD units):", MDE_SD)
led = []
for _, r in PT[PT.program.isin(HEAD) & PT.group.isin(GROUPS) & PT.effect.isin(["PSCA", "direct RT", "indirect RT", "DiD"])].iterrows():
    so = SO[(SO.group == r.group) & (SO.program == r.program) & (SO.effect == r.effect)]
    m2 = bool(so.method2_ok.iloc[0]) if len(so) else False
    m2s = so.method2_sign.iloc[0] if len(so) else np.nan
    if not m2 and r.family != "chemokine":                     # small curated modules: gene-level route
        gl = PROGRAMS[r.program]["genes"]
        rep = REPL[(REPL.group == r.group) & (REPL.effect == r.effect) & REPL.gene.isin(gl) & REPL.replicated]
        if len(rep) and len(PROGRAMS[r.program]["genes"]) < 10:
            m2, m2s = True, np.sign(rep.lfc_1.mean())
    lo = LOO[(LOO.group == r.group) & (LOO.program == r.program) & (LOO.effect == r.effect)]
    flips = int(lo.loo_flips.iloc[0]) if len(lo) else np.nan
    amb = ambient_flag(r.group, r.program)
    main, pm, agree = r.get("main"), r.get("p_main"), r.get("sign_agree")
    same = pd.notna(main) and pd.notna(m2s) and np.sign(main) == m2s
    if (m2 and same and agree == True and pm < 0.05 and flips == 0 and not amb
            and (pd.isna(r.rand_pct) or r.rand_pct >= 0.95)):
        tier = "Definitive"
    elif agree == True and ((pd.notna(pm) and pm < 0.10) or m2):
        tier = "Supported"
    elif (pd.notna(r.get("p_inter")) and r.p_inter < 0.05) or \
         any(pd.notna(r.get(c)) and r.get(c) < 0.05 for c in ["p_nonRT", "p_RT", "p_PSCA", "p_TAG72"]):
        tier = "Suggestive"
    else:
        tier = "No evidence"
    led.append(dict(family=r.family, program=r.program, group=r.group, effect=r.effect, tier=tier,
                    score_main=main, score_p=pm, sign_agree=agree, d_1=r.get("d_1"), d_2=r.get("d_2"),
                    CARxRT=r.get("interaction"), p_CARxRT=r.get("p_inter"), second_method=m2,
                    loo_flips=flips, ambient_suspect=amb, rand_pct=r.rand_pct,
                    MDE_SD=MDE_SD.get(r.effect)))
LED = pd.DataFrame(led)
# cytotoxicity sets scored in cells that are not cytotoxic lymphocytes read MHC / Fc-receptor genes,
# not killing (GO T-cell cytotoxicity is 26/57 MHC genes) — reclassify instead of claiming cytotoxicity
LYMPH = {"NK", "CD8_T", "cytotoxic_lymph"}
nonlym = (LED.family == "cytotoxicity") & ~LED.group.isin(LYMPH)
LED["note"] = ""
LED.loc[nonlym, "note"] = "not a cytotoxic readout in this cell type (MHC / Fc-receptor genes drive the set)"
LED.loc[nonlym & LED.tier.isin(["Definitive", "Supported", "Suggestive"]), "tier"] = "Reclassified"
# the same logic for exhaustion: Entpd1, Cd38, Nr4a1-3, Batf, Irf4 are activation / IFN / immediate-early
# genes in myeloid and tumour cells — an "exhaustion" score there is not T-cell exhaustion
nonT = (LED.family == "exhaustion") & ~LED.group.isin(LYMPH)
LED.loc[nonT, "note"] = "not an exhaustion readout outside T / NK cells (activation / immediate-early genes)"
LED.loc[nonT & LED.tier.isin(["Definitive", "Supported", "Suggestive"]), "tier"] = "Reclassified"
# ambient-suspect claims in non-macrophage compartments get their own tier rather than "Supported"
amb_rows = LED.ambient_suspect & LED.tier.isin(["Supported", "Suggestive"])
LED.loc[amb_rows, "note"] = "majority of the program's signal in this compartment is ambient macrophage RNA (01 §4)"
LED.loc[amb_rows, "tier"] = "Ambient-suspect"
LED.to_csv(OUT/"ledger_programs.csv", index=False)

TIER_ORDER = ["Definitive", "Supported", "Suggestive", "No evidence", "Reclassified", "Ambient-suspect"]
print(pd.crosstab([LED.family, LED.effect], pd.Categorical(LED.tier, TIER_ORDER)).to_string())
for eff in ["PSCA", "direct RT", "indirect RT", "DiD"]:
    d = LED[(LED.effect == eff) & LED.tier.isin(["Definitive", "Supported"])]
    print(f"\n=== {eff.upper()} — Definitive / Supported ({len(d)}) ===")
    print(d.sort_values(["tier", "family", "group"])[["tier", "family", "program", "group", "score_main", "score_p",
                                                      "d_1", "d_2", "loo_flips", "ambient_suspect", "rand_pct"]]
            .round(3).to_string(index=False) if len(d) else "    none")
d = LED[(LED.p_CARxRT < 0.05)]
print(f"\n=== CAR x RT interactions on program scores (p < 0.05): {len(d)} ===")
print(d[["family", "program", "group", "effect", "d_1", "d_2", "CARxRT", "p_CARxRT", "ambient_suspect"]]
        .round(3).to_string(index=False) if len(d) else "    none")

MDE (SD units): {'PSCA': np.float64(1.5), 'direct RT': np.float64(2.1), 'indirect RT': np.float64(2.1), 'DiD': np.float64(2.1)}


col_0                     Definitive  Supported  Suggestive  No evidence  Reclassified  Ambient-suspect
family       effect                                                                                    
IFN          DiD                   0          1           0           17             0                0
             PSCA                 11          5           0            6             0                0
             direct RT             0          0           0           22             0                0
             indirect RT           1          6           0           13             0                0
MHC-I        DiD                   0          0           0           18             0                0
             PSCA                  5          5           0           12             0                0
             direct RT             0          0           0           22             0                0
             indirect RT           0          2           1     

## 5. Exhaustion ledger (T and NK cells; sample-level, with MDE)

In [5]:
el = []
for _, r in EX.iterrows():
    mde = EMDE[(EMDE["pop"] == r["pop"]) & (EMDE.measure == r.measure)]
    pm, agree = r.get("p_main"), r.get("sign_agree")
    tier = ("Supported" if agree == True and pd.notna(pm) and pm < 0.05 else
            "Suggestive" if (pd.notna(pm) and pm < 0.10) or (pd.notna(r.get("p_inter")) and r.p_inter < 0.05) else
            "No evidence")
    el.append(dict(pop=r["pop"], measure=r.measure, effect=r.effect, tier=tier, main=r.get("main"), p_main=pm,
                   sign_agree=agree, CARxRT=r.get("interaction"), p_CARxRT=r.get("p_inter"),
                   MDE=float(mde.MDE.iloc[0]) if len(mde) else np.nan,
                   mean=float(mde["mean"].iloc[0]) if len(mde) else np.nan))
ELED = pd.DataFrame(el)
ELED.to_csv(OUT/"ledger_exhaustion.csv", index=False)
print("exhaustion measures are never 'Definitive' here: no second independent method is powered "
      "at ~14 CD8 cells per sample.")
print(ELED[ELED.tier != "No evidence"].round(3).to_string(index=False) or "no measure above 'No evidence'")
print("\nCD8_T measures with no evidence — MDE vs mean (MDE larger than the mean = could not have seen it):")
print(ELED[(ELED["pop"] == "CD8_T") & (ELED.tier == "No evidence") & (ELED.effect == "PSCA")]
      [["measure", "mean", "MDE"]].round(3).to_string(index=False))

exhaustion measures are never 'Definitive' here: no second independent method is powered at ~14 CD8 cells per sample.
  pop                  measure      effect       tier   main  p_main sign_agree  CARxRT  p_CARxRT   MDE  mean
CD8_T                   IR_ge2   direct RT  Supported  0.173   0.030       True  -0.033     0.920 0.454 0.484
CD8_T                Tox_Pdcd1        PSCA Suggestive  0.083   0.052       True  -0.113     0.234 0.166 0.113
CD8_T                   Pdcd1+        PSCA Suggestive  0.087   0.062       True  -0.105     0.339 0.201 0.200
CD8_T                  Havcr2+   direct RT  Supported  0.089   0.040       True   0.092     0.370 0.264 0.154
CD8_T                    Lag3+        PSCA  Supported  0.073   0.005       True   0.066     0.257 0.154 0.079
CD8_T                    Lag3+   direct RT  Supported  0.077   0.030       True   0.102     0.200 0.154 0.079
CD8_T                     Tox+   direct RT Suggestive -0.181   0.070       True  -0.109     0.590 0.317 0.503
CD

## 6. The chemokine ledger — ligands and receptors in separate rows

In [6]:
CH_L = {g: a for a, v in AXES.items() for g in v["ligands"]}
CH_R = {g: a for a, v in AXES.items() for g in v["receptors"]}
pts = PT[PT.family == "chemokine"]
cl = []
for _, r in REPL[REPL.gene.isin(list(CH_L) + list(CH_R)) & REPL.group.isin(GROUPS + ["Endothelial", "Fibroblast"])].iterrows():
    role = "ligand" if r.gene in CH_L else "receptor"
    axis = CH_L.get(r.gene, CH_R.get(r.gene))
    sc = pts[(pts.group == r.group) & (pts.program == f"{axis}_{role}s") & (pts.effect == r.effect)]
    sc_ok = len(sc) and sc.sign_agree.iloc[0] == True and sc.p_main.iloc[0] < 0.05 and \
        np.sign(sc["main"].iloc[0]) == np.sign(r.lfc_1)
    lf = str(r.loo_flips)
    loo_ok = lf.startswith("0/") and "| 0/" in lf
    amb = (r.group in NON_MYELOID) and ((r.group, r.gene) in amb_genes.index) and amb_genes[(r.group, r.gene)] < 0.5
    if r.replicated and loo_ok and not amb and r.baseMean >= 20 and sc_ok:
        tier = "Definitive"
    elif r.replicated:
        tier = "Supported"
    elif min(r.q_1, r.q_2) < 0.05:
        tier = "Suggestive"
    else:
        tier = "No evidence"
    cl.append(dict(axis=axis, priority=AXES[axis]["priority"], role=role, gene=r.gene, group=r.group,
                   effect=r.effect, observed_transcriptional_change=tier, lfc_1=r.lfc_1, q_1=r.q_1, lfc_2=r.lfc_2,
                   q_2=r.q_2, baseMean=r.baseMean, loo=lf, ambient_suspect=amb,
                   axis_score_main=sc["main"].iloc[0] if len(sc) else np.nan,
                   axis_score_p=sc.p_main.iloc[0] if len(sc) else np.nan))
for _, r in PANEL[PANEL.gene.isin(list(CH_L) + list(CH_R)) & PANEL.contrast.isin(["CARxRT", "DiD_PSCA", "DiD_TAG72"])
                  & (PANEL.panel_padj < 0.05)].iterrows():
    role = "ligand" if r.gene in CH_L else "receptor"; axis = CH_L.get(r.gene, CH_R.get(r.gene))
    cl.append(dict(axis=axis, priority=AXES[axis]["priority"], role=role, gene=r.gene, group=r.group,
                   effect=r.contrast, observed_transcriptional_change="Suggestive", lfc_1=r.log2FoldChange,
                   q_1=r.panel_padj, baseMean=r.baseMean))
CLED = pd.DataFrame(cl)

# inferred chemotactic potential: from 04b's axis table, capped at Suggestive without a population association
pot = AXT.groupby(["ligand", "source", "effect"]).agg(
    assoc_q=("population_q", "min"), assoc_r=("population_partial_r", lambda v: v.loc[v.abs().idxmax()] if v.notna().any() else np.nan),
    targets=("target", lambda v: ",".join(sorted(set(v)))),
    receptor_measurable=("receptor_measurable", "max")).reset_index()


def infer(r):
    if r.role != "ligand": return "n/a (receptor row)"
    p = pot[(pot.ligand == r.gene) & (pot.source == r.group) & (pot.effect == r.effect)]
    if p.empty: return "not assessed (no population pair for this source)"
    p = p.iloc[0]
    if not p.receptor_measurable: return "ligand-only: receptor unmeasurable"
    if r.observed_transcriptional_change in ("Definitive", "Supported") and p.assoc_q < 0.10:
        return f"Supported association with {p.targets} composition (r={p.assoc_r:+.2f}) — not recruitment"
    if r.observed_transcriptional_change in ("Definitive", "Supported"):
        return "Suggestive: ligand change without a population association"
    return "None"


CLED["inferred_chemotactic_potential"] = CLED.apply(infer, axis=1)
# sensitivity columns: does dropping Ccl5 / Xcl1 change the cytotoxic / progenitor modules in this group?
sens = SENS.set_index(["group", "effect", "program"])
CLED["Ccl5_sensitivity"] = [("flips cyto_effector" if sens.loc[(r.group, r.effect, "cyto_effector"), "flip"] else "no flip")
                            if r.gene == "Ccl5" and (r.group, r.effect, "cyto_effector") in sens.index else ""
                            for _, r in CLED.iterrows()]
CLED["Xcl1_sensitivity"] = [("flips exh_progenitor" if sens.loc[(r.group, r.effect, "exh_progenitor"), "flip"] else "no flip")
                            if r.gene == "Xcl1" and (r.group, r.effect, "exh_progenitor") in sens.index else ""
                            for _, r in CLED.iterrows()]
CLED.to_csv(OUT/"ledger_chemokines.csv", index=False)
print(pd.crosstab([CLED.role, CLED.effect], pd.Categorical(CLED.observed_transcriptional_change, TIER_ORDER)).to_string())
show = CLED[CLED.observed_transcriptional_change.isin(["Definitive", "Supported", "Suggestive"])]
print(show.sort_values(["observed_transcriptional_change", "axis", "role"])
      [["observed_transcriptional_change", "axis", "role", "gene", "group", "effect", "lfc_1", "q_1", "lfc_2", "q_2",
        "baseMean", "loo", "ambient_suspect", "inferred_chemotactic_potential"]].round(3).to_string(index=False))

col_0                 Definitive  Supported  Suggestive  No evidence
role     effect                                                     
ligand   DiD_PSCA              0          0           2            0
         DiD_TAG72             0          0           5            0
         PSCA                 31          4          29          117
         direct RT             0          0           4          157
         indirect RT           1          1          17          116
receptor CARxRT                0          0           1            0
         DiD_PSCA              0          0           1            0
         PSCA                  0          3          10           62
         direct RT             0          0           3           65
         indirect RT           0          0           4           50
observed_transcriptional_change  axis     role   gene              group      effect  lfc_1   q_1  lfc_2   q_2  baseMean         loo ambient_suspect                        

## 7. Summary — the definitive statements

In [7]:
print("=" * 100); print("SUMMARY — report these back"); print("=" * 100)
print("\n[ambient check on the tumour MHC-II CAR x RT interaction]")
print(AMBX.to_string(index=False))
for eff in ["PSCA", "direct RT", "indirect RT", "DiD"]:
    d = LED[(LED.effect == eff) & (LED.tier == "Definitive")]
    s = LED[(LED.effect == eff) & (LED.tier == "Supported")]
    print(f"\n[{eff}] DEFINITIVE ({len(d)}): " + ", ".join(f"{r.program}@{r.group}({r.score_main:+.1f})" for _, r in d.iterrows()))
    print(f"         supported ({len(s)}): " + ", ".join(f"{r.program}@{r.group}" for _, r in s.iterrows()))
print("\n[chemokines — observed transcriptional change, Definitive]")
for _, r in CLED[CLED.observed_transcriptional_change == "Definitive"].iterrows():
    print(f"    {r.effect:<11s} {r.role:<8s} {r.gene:<7s} @ {r.group:<18s} {r.lfc_1:+.1f}/{r.lfc_2:+.1f}  "
          f"-> {r.inferred_chemotactic_potential}")
print("[chemokines — Supported]")
for _, r in CLED[CLED.observed_transcriptional_change == "Supported"].iterrows():
    print(f"    {r.effect:<11s} {r.role:<8s} {r.gene:<7s} @ {r.group:<18s} {r.lfc_1:+.1f}/{r.lfc_2:+.1f}")
print(f"\n[program ledger] {len(LED)} claims; tiers: {LED.tier.value_counts().to_dict()}")
print(f"[exhaustion, T/NK] {len(ELED)} measure x effect tests; Supported {int((ELED.tier == 'Supported').sum())} "
      f"(~{0.025*len(ELED):.0f} expected by chance at sign-agree p<0.05), Suggestive "
      f"{int((ELED.tier == 'Suggestive').sum())}. Measures share genes, so hits cluster.")
for (pp, eff), d in ELED[ELED.tier == "Supported"].groupby(["pop", "effect"]):
    print(f"    {pp:<16s} {eff:<11s} " + ", ".join(f"{r.measure}({r.main:+.2f})" for _, r in d.iterrows()))
print(f"\n[files] {OUT}: ledger_programs.csv, ledger_exhaustion.csv, ledger_chemokines.csv, "
      f"ledger_tumour_mhc_ambient_check.csv")
print("=" * 100)

SUMMARY — report these back

[ambient check on the tumour MHC-II CAR x RT interaction]
                            measure              Tu1              Tu2
log2 rho (macrophage contamination)  +1.45 (p=0.165)  +0.31 (p=0.755)
                    H2-Ab1 observed  +2.12 (p=0.045)  +2.38 (p=0.025)
          H2-Ab1 ambient-subtracted  +2.85 (p=0.250)  +3.08 (p=0.020)
     H2-Ab1 clean tumour cells only +0.151 (p=0.415) +0.095 (p=0.445)
                    H2-Eb1 observed  +2.36 (p=0.055)  +2.83 (p=0.030)
          H2-Eb1 ambient-subtracted  +3.33 (p=0.180)  +4.74 (p=0.005)
     H2-Eb1 clean tumour cells only +0.357 (p=0.100) +0.135 (p=0.370)
                      Cd74 observed  +2.10 (p=0.040)  +2.80 (p=0.045)
            Cd74 ambient-subtracted  +2.39 (p=0.395)  +6.05 (p=0.005)
       Cd74 clean tumour cells only +0.163 (p=0.655) +0.216 (p=0.260)
                     H2-Aa observed  +2.03 (p=0.080)  +2.02 (p=0.105)
           H2-Aa ambient-subtracted  +2.93 (p=0.235)  +3.80 (p=0.025)
   